# Data Translation Challenge: Pumpkinmeter Recommendation Engine
### OMSBA 5240 | Seattle University
### Linh Le

**Use Case:** Building a collaborative-filtering movie recommender for Ripe Pumpkins' Pumpkinmeter score, using Apache Spark MLlib's ALS algorithm on the full MovieLens "latest" dataset (per GroupLens' current listing: ~33,000,000 ratings / 86,000 movies / 330,975 users; counts grow over time since the download link is kept stable, so this is larger than the 27M/58K/280K figures in the assignment brief).

**Tutorial reference:** https://www.codementor.io/jadianes/building-a-recommender-with-apache-spark-python-example-app-part1-du1083qbw

**Dataset source:** https://grouplens.org/datasets/movielens/ | **Source platform:** https://movielens.org/

## 1. Setup: SparkContext

In [ ]:
import pyspark
sc = pyspark.SparkContext('local[*]')

## 2. File Download and Extraction

**Note:** Per the debugging tips, the small and complete MovieLens datasets have already been downloaded and extracted into the `work` folder on this EC2 instance (`ml-latest` and `ml-latest-small`). The two code blocks below (`Getting file(s)` and `Extracting file(s)`) are **skipped** — do not re-run them. They're left here only to show where the datasets came from.

In [ ]:
# File download - ALREADY DONE, DO NOT RE-RUN
complete_dataset_url = 'http://files.grouplens.org/datasets/movielens/ml-latest.zip'
small_dataset_url = 'http://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

import os
datasets_path = os.path.join('/home/jovyan', 'work')
complete_dataset_path = os.path.join(datasets_path, 'ml-latest.zip')
small_dataset_path = os.path.join(datasets_path, 'ml-latest-small.zip')

# SKIP - Getting file(s)
# import urllib.request
# small_f = urllib.request.urlretrieve (small_dataset_url, small_dataset_path)
# complete_f = urllib.request.urlretrieve (complete_dataset_url, complete_dataset_path)

# SKIP - Extracting file(s)
# import zipfile
# with zipfile.ZipFile(small_dataset_path, "r") as z:
#     z.extractall(datasets_path)
# with zipfile.ZipFile(complete_dataset_path, "r") as z:
#     z.extractall(datasets_path)

### 2b. Auto-resolve dataset folder paths

Rather than hardcoding a path and hoping it's right, this cell searches a few likely locations for `ratings.csv` / `movies.csv` under both `ml-latest` and `ml-latest-small` (handling the common case where the zip extracts one folder level deeper than expected, e.g. `ml-latest/ml-latest/ratings.csv`). It raises a clear error telling you exactly where it looked if it can't find the files, instead of the cryptic `Py4JJavaError` from before.

In [ ]:
import os

def find_dataset_folder(base_path, folder_name):
    """Return the first path under base_path that actually contains
    ratings.csv and movies.csv for the given MovieLens folder name."""
    candidates = [
        os.path.join(base_path, folder_name),
        os.path.join(base_path, folder_name, folder_name),
    ]
    for candidate in candidates:
        if os.path.isfile(os.path.join(candidate, 'ratings.csv')) and \
           os.path.isfile(os.path.join(candidate, 'movies.csv')):
            return candidate

    # Last resort: walk the tree looking for ratings.csv near this folder name
    for root, dirs, files in os.walk(base_path):
        if folder_name in root and 'ratings.csv' in files and 'movies.csv' in files:
            return root

    raise FileNotFoundError(
        "Could not find ratings.csv/movies.csv for '{}' under {}. "
        "Checked: {}. Run os.listdir() on {} to see what's actually there, "
        "and confirm the dataset was downloaded and extracted.".format(
            folder_name, base_path, candidates, base_path)
    )

print('Contents of work folder:', os.listdir(datasets_path))

small_dataset_dir = find_dataset_folder(datasets_path, 'ml-latest-small')
complete_dataset_dir = find_dataset_folder(datasets_path, 'ml-latest')

print('Resolved small dataset folder:', small_dataset_dir)
print('Resolved complete dataset folder:', complete_dataset_dir)

## 3. Loading and Parsing Datasets

We load both the **small** dataset (used only to select the best ALS rank quickly) and the **complete/full** dataset (used to build the actual recommender model, per the assignment requirement).

In [ ]:
# Small dataset - ratings (used for fast parameter selection)
small_ratings_file = os.path.join(small_dataset_dir, 'ratings.csv')

small_ratings_raw_data = sc.textFile(small_ratings_file)
small_ratings_raw_data_header = small_ratings_raw_data.take(1)[0]

small_ratings_data = small_ratings_raw_data \
    .filter(lambda line: line != small_ratings_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (tokens[0], tokens[1], tokens[2])).cache()

small_ratings_data.take(3)

In [ ]:
# Small dataset - movies
small_movies_file = os.path.join(small_dataset_dir, 'movies.csv')

small_movies_raw_data = sc.textFile(small_movies_file)
small_movies_raw_data_header = small_movies_raw_data.take(1)[0]

small_movies_data = small_movies_raw_data \
    .filter(lambda line: line != small_movies_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (tokens[0], tokens[1])).cache()

small_movies_data.take(3)

In [ ]:
# Complete/full dataset - ratings
complete_ratings_file = os.path.join(complete_dataset_dir, 'ratings.csv')

complete_ratings_raw_data = sc.textFile(complete_ratings_file)
complete_ratings_raw_data_header = complete_ratings_raw_data.take(1)[0]

complete_ratings_data = complete_ratings_raw_data \
    .filter(lambda line: line != complete_ratings_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), int(tokens[1]), float(tokens[2]))).cache()

print("There are {} recommendations in the complete dataset".format(complete_ratings_data.count()))

In [ ]:
# Complete/full dataset - movies
complete_movies_file = os.path.join(complete_dataset_dir, 'movies.csv')

complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

complete_movies_data = complete_movies_raw_data \
    .filter(lambda line: line != complete_movies_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), tokens[1], tokens[2])).cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]), x[1]))

print("There are {} movies in the complete dataset".format(complete_movies_titles.count()))

In [ ]:
def get_counts_and_averages(ID_and_ratings_tuple):
    nratings = len(ID_and_ratings_tuple[1])
    return ID_and_ratings_tuple[0], (nratings, float(sum(x for x in ID_and_ratings_tuple[1])) / nratings)

movie_ID_with_ratings_RDD = (complete_ratings_data.map(lambda x: (x[1], x[2])).groupByKey())
movie_ID_with_avg_ratings_RDD = movie_ID_with_ratings_RDD.map(get_counts_and_averages)
movie_rating_counts_RDD = movie_ID_with_avg_ratings_RDD.map(lambda x: (x[0], x[1][0]))

## 4. Selecting ALS Parameters (Rank) on the Small Dataset

We split the small dataset into training/validation/test and try a few candidate ranks, picking the one that minimizes RMSE on the validation set.

In [ ]:
training_RDD, validation_RDD, test_RDD = small_ratings_data.randomSplit([6, 2, 2], seed=0)
validation_for_predict_RDD = validation_RDD.map(lambda x: (x[0], x[1]))
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))

In [ ]:
from pyspark.mllib.recommendation import ALS
import math

seed = 5
iterations = 10
regularization_parameter = 0.1
ranks = [4, 8, 12]
errors = [0, 0, 0]
err = 0
tolerance = 0.02

min_error = float('inf')
best_rank = -1
best_iteration = -1
for rank in ranks:
    model = ALS.train(training_RDD, rank, seed=seed, iterations=iterations,
                       lambda_=regularization_parameter)
    predictions = model.predictAll(validation_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
    rates_and_preds = validation_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
    error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())
    errors[err] = error
    err += 1
    print('For rank %s the RMSE is %s' % (rank, error))
    if error < min_error:
        min_error = error
        best_rank = rank

print('The best model was trained with rank %s' % best_rank)

## 5. Building the Recommender Model on the Complete/Full Dataset

Per the assignment requirements, the actual recommender used for the CEO presentation is trained on the **complete/full** dataset (~33M ratings per GroupLens' current listing), using the best rank found above.

In [ ]:
training_RDD, test_RDD = complete_ratings_data.randomSplit([7, 3], seed=0)

complete_model = ALS.train(training_RDD, best_rank, seed=seed,
                            iterations=iterations, lambda_=regularization_parameter)

In [ ]:
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))

predictions = complete_model.predictAll(test_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
rates_and_preds = test_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())

print('For testing data the RMSE is %s' % (error))

## 6. Helper Function: Add New User, Retrain, Recommend

This function takes a list of `(0, movieID, rating)` tuples for a new user (userID always `0`), unions them with the complete dataset, retrains the ALS model, and returns the top-N recommended movies filtered to only movies with at least `min_ratings` ratings — exactly the "Adding new user ratings" workflow from the tutorial, wrapped so we can run it for both users and both scenarios without repeating code.

In [ ]:
def get_top_recommendations(user_ratings, min_ratings, top_n, label):
    new_user_ID = 0
    new_user_ratings_RDD = sc.parallelize(user_ratings)
    complete_data_with_new_ratings_RDD = complete_ratings_data.union(new_user_ratings_RDD)

    new_ratings_model = ALS.train(complete_data_with_new_ratings_RDD, best_rank, seed=seed,
                                   iterations=iterations, lambda_=regularization_parameter)

    new_user_ratings_ids = map(lambda x: x[1], user_ratings)
    new_user_unrated_movies_RDD = (complete_movies_data
                                    .filter(lambda x: x[0] not in new_user_ratings_ids)
                                    .map(lambda x: (new_user_ID, x[0])))

    new_user_recommendations_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)

    new_user_recommendations_rating_RDD = new_user_recommendations_RDD.map(lambda x: (x.product, x.rating))
    new_user_recommendations_rating_title_and_count_RDD = \
        new_user_recommendations_rating_RDD.join(complete_movies_titles).join(movie_rating_counts_RDD)

    new_user_recommendations_rating_title_and_count_RDD = \
        new_user_recommendations_rating_title_and_count_RDD.map(lambda r: (r[1][0][1], r[1][0][0], r[1][1]))

    top_movies = new_user_recommendations_rating_title_and_count_RDD \
        .filter(lambda r: r[2] >= min_ratings).takeOrdered(top_n, key=lambda x: -x[1])

    print('{} - TOP {} recommended movies (with >= {} ratings):\n{}'.format(
        label, top_n, min_ratings, '\n'.join(map(str, top_movies))))

    return top_movies

## 7. New User Ratings

**User 1 (Linh):** 10 movies, ratings 3-5 (reused from the movie-recommendation assignment).

**User 2 (friend/family):** 10 movies, ratings 2-3.

In [ ]:
# User 1 - Linh
user1_ratings = [
     (0,1,4),    # Toy Story (1995)
     (0,364,5),  # Lion King, The (1994)
     (0,588,4),  # Aladdin (1992)
     (0,595,4),  # Beauty and the Beast (1991)
     (0,2085,3), # 101 Dalmatians (1961)
     (0,2394,3), # Prince of Egypt, The (1998)
     (0,3000,5), # Princess Mononoke (1997)
     (0,4306,5), # Shrek (2001)
     (0,4886,4), # Monsters, Inc. (2001)
     (0,4896,4)  # Harry Potter and the Sorcerer's Stone (2001)
    ]
print('User 1 ratings: {}'.format(user1_ratings))

In [ ]:
# User 2 - friend/family member
user2_ratings = [
     (0,48,3),   # Pocahontas (1995)
     (0,261,2),  # Little Women (1994)
     (0,1013,3), # Parent Trap, The (1961)
     (0,1032,2), # Alice in Wonderland (1951)
     (0,2087,3), # Peter Pan (1953)
     (0,2096,2), # Sleeping Beauty (1959)
     (0,4369,3), # Fast and the Furious, The (2001)
     (0,4447,2), # Legally Blonde (2001)
     (0,4701,3), # Rush Hour 2 (2001)
     (0,6377,3)  # Finding Nemo (2003)
    ]
print('User 2 ratings: {}'.format(user2_ratings))

## 8. Running the 4 Cases

- **User 1 – Scenario 1:** full dataset, filter out movies with fewer than 25 ratings
- **User 1 – Scenario 2:** full dataset, filter out movies with fewer than 100 ratings
- **User 2 – Scenario 1:** full dataset, filter out movies with fewer than 25 ratings
- **User 2 – Scenario 2:** full dataset, filter out movies with fewer than 100 ratings

Each case returns the top 15 recommended movies.

In [1]:
user1_scenario1 = get_top_recommendations(user1_ratings, min_ratings=25, top_n=15, label='User 1 - Scenario 1')

User 1 - Scenario 1 - TOP 15 recommended movies (with >= 25 ratings):
('Wallace & Gromit: The Best of Aardman Animation (1996)', 4.710160110568713, 27)
('"Shawshank Redemption', 4.70000679324299, 317)
('Patton (1970)', 4.624273012170483, 33)
('"Philadelphia Story', 4.589424945342907, 29)
('Star Wars: Episode IV - A New Hope (1977)', 4.584611452101444, 251)
("Schindler's List (1993)", 4.577685979089203, 220)
('Star Wars: Episode V - The Empire Strikes Back (1980)', 4.560519884422914, 211)
('"Great Escape', 4.5574577372484395, 43)
('Lawrence of Arabia (1962)', 4.553266194160184, 45)
('Wallace & Gromit: The Wrong Trousers (1993)', 4.548450008402453, 56)
('"Princess Bride', 4.548050619691201, 142)
('Amadeus (1984)', 4.546586041273068, 76)
('Life Is Beautiful (La Vita è bella) (1997)', 4.543643264973439, 88)
('"Godfather', 4.5342251888186285, 192)
('Casablanca (1942)', 4.530111516252315, 100)

In [1]:
user1_scenario2 = get_top_recommendations(user1_ratings, min_ratings=100, top_n=15, label='User 1 - Scenario 2')

User 1 - Scenario 2 - TOP 15 recommended movies (with >= 100 ratings):
('"Shawshank Redemption', 4.70000679324299, 317)
('Star Wars: Episode IV - A New Hope (1977)', 4.584611452101444, 251)
("Schindler's List (1993)", 4.577685979089203, 220)
('Star Wars: Episode V - The Empire Strikes Back (1980)', 4.560519884422914, 211)
('"Princess Bride', 4.548050619691201, 142)
('"Godfather', 4.5342251888186285, 192)
('Casablanca (1942)', 4.530111516252315, 100)
('"Usual Suspects', 4.523111233751232, 204)
('Monty Python and the Holy Grail (1975)', 4.5147822774421, 136)
('Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)', 4.504726058253806, 200)
('"Amelie (Fabuleux destin d\'Amélie Poulain', 4.49888932450791, 120)
('"Lord of the Rings: The Return of the King', 4.47196017489378, 185)
('Fight Club (1999)', 4.471168226911484, 218)
('"Godfather: Part II', 4.471059105178089, 129)
('"Lord of the Rings: The Fellowship of the Ring', 4.4587277497817075, 198)

In [1]:
user2_scenario1 = get_top_recommendations(user2_ratings, min_ratings=25, top_n=15, label='User 2 - Scenario 1')

User 2 - Scenario 1 - TOP 15 recommended movies (with >= 25 ratings):
('"Grand Day Out with Wallace and Gromit', 3.3893466393268747, 28)
('Eternal Sunshine of the Spotless Mind (2004)', 3.3450505885002793, 131)
('Cinema Paradiso (Nuovo cinema Paradiso) (1989)', 3.3072721084964742, 34)
('"Wolf of Wall Street', 3.2951346054064707, 54)
('3:10 to Yuma (2007)', 3.288245992984778, 25)
('"Apartment', 3.278225110753463, 27)
('Good Will Hunting (1997)', 3.2691192773734747, 141)
('Taken (2008)', 3.2528206800881794, 42)
('Star Wars: Episode VII - The Force Awakens (2015)', 3.2445788810823495, 41)
('Guardians of the Galaxy 2 (2017)', 3.2218582715711364, 27)
('Little Miss Sunshine (2006)', 3.215051274371784, 77)
('Shutter Island (2010)', 3.2103945053515126, 67)
('"Dark Knight', 3.2080088817282766, 149)
('American History X (1998)', 3.204430464024259, 129)
('Argo (2012)', 3.2021452514865927, 28)

In [1]:
user2_scenario2 = get_top_recommendations(user2_ratings, min_ratings=100, top_n=15, label='User 2 - Scenario 2')

User 2 - Scenario 2 - TOP 15 recommended movies (with >= 100 ratings):
('Eternal Sunshine of the Spotless Mind (2004)', 3.3450505885002793, 131)
('Good Will Hunting (1997)', 3.2691192773734747, 141)
('"Dark Knight', 3.2080088817282766, 149)
('American History X (1998)', 3.204430464024259, 129)
('Fight Club (1999)', 3.1920880318785496, 218)
('"Matrix', 3.1880944235387765, 278)
('"Shawshank Redemption', 3.1849741596381245, 317)
('Forrest Gump (1994)', 3.1758471106145763, 329)
('"Departed', 3.154217854121116, 107)
('Trainspotting (1996)', 3.148500116683905, 102)
('"Green Mile', 3.1356415576196754, 111)
('WALL·E (2008)', 3.1335446838798253, 104)
('"Beautiful Mind', 3.1300396388182232, 123)
('Braveheart (1995)', 3.1198024987618687, 237)
('Up (2009)', 3.1165025591340942, 105)

**Note on movie titles above:** several titles are cut short (e.g. `'"Shawshank Redemption'`, `'"Godfather'`). This is a parsing artifact, not a data or model problem: `movies.csv` wraps titles that contain a comma in quotes (e.g. `"Shawshank Redemption, The (1994)"`, since MovieLens moves the article to the end), but our simple `line.split(",")` parsing doesn't respect those quotes, so it splits the title at the internal comma. The recommendation itself (predicted rating, rating count) is unaffected — only the display of the title is. The full corrected titles are used in the Word report and slide deck.

## 9. Insights / Foresights

**Scenario 1 (≥ 25 ratings) surfaces niche/cult titles; Scenario 2 (≥ 100 ratings) surfaces mainstream classics.**
For User 1, Scenario 1 includes cult and lower-visibility titles like *Wallace & Gromit: The Best of Aardman Animation* (27 ratings), *Patton* (33), and *The Philadelphia Story* (29). Raising the threshold to 100 knocks all of these out, replacing them with widely-seen blockbusters and canonical classics (*Star Wars*, *The Godfather*, *The Lord of the Rings*, *Fight Club*). The same pattern holds for User 2. This is a direct trade-off between **discovery** (surfacing hidden gems a user might not find otherwise) and **reliability** (recommending titles with a long track record of broad approval).

**The model matches rating *behavior*, not genre.**
User 1 rated only animated/family films (Toy Story, Lion King, Shrek, Harry Potter) highly, yet the top recommendations are almost entirely adult dramas and classics (Shawshank, Schindler's List, Casablanca, The Godfather). This shows that ALS collaborative filtering finds users with *similar rating patterns* across the whole catalog, not similar genres — people who rate beloved family films highly also tend to rate other beloved, critically-acclaimed films highly. For the board, this is an important nuance: the Pumpkinmeter won't necessarily recommend "more of the same genre" — it recommends "what similar tastes also loved," which can look surprising to an end user without some framing/explanation in the product UI.

**User 2's lower ratings compress the whole recommendation scale.**
User 2 rated everything in the 2-3 range, and their predicted ratings for recommended movies max out around 3.1-3.4 (versus User 1's 4.5-4.7). ALS is picking up on User 2's general tendency to rate things lower, not just which specific movies they'd enjoy. Business implication: a single global "Pumpkinmeter" cutoff for what counts as a "recommended" movie would systematically under-recommend to naturally harsher raters like User 2, unless the score is normalized per-user rather than compared on a raw absolute scale.

**Business recommendation:** consider giving users (or Ripe Pumpkins' product team) a toggle between a "Discover" mode (lower ratings-count threshold, more niche titles, higher risk of a miss) and a "Safe Bet" mode (higher threshold, mainstream/critically-loved titles, lower personalization but higher confidence) rather than picking one fixed threshold for everyone.